# Controlled detrending with consistent standardization

This controlled experiment evaluates whether the pipeline's conditional detrending procedure reduces the classification-performance gap between official SPOC/QLP products and matched TESSCut light curves. Raw TESSCut light curves are conditionally detrended, passed through the same pipeline standardization, and processed with the same feature extractor. Baseline stars use their existing standardized `lightCurvePath`, uncorrected stars retain that path, and official products and original flux-scale metadata are held fixed.

The pipeline's raw reader, standardizer, FITS writer, drift detection, correction, and feature-extraction procedures are reproduced directly in this notebook. The experiment includes post-detrending standardization so that original and corrected TESSCut light curves undergo the same standardization procedure before feature extraction.

Two validation checks precede model evaluation: extraction from the original standardized paths reproduces the saved baseline features, and raw → standardization → extraction without detrending reproduces the same 39 predictors. Both checks passed for the SPOC- and QLP-matched cohorts. The analysis preserves both matched cohorts and all 100 paired train/test splits and requires no external-service queries.


In [1]:
from __future__ import annotations
import json, logging, hashlib, warnings
import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
from astropy.io import fits
from astropy.timeseries import LombScargle
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from IPython.display import display
logging.basicConfig(level=logging.INFO)
Logger = logging.getLogger('ControlledConditionalDetrending')
Project_Root = Path(os.environ.get('Project_Root', '/data/projects/TESS_Research_RASTI_Publication'))
RawDataRoot = Project_Root / 'data' / 'raw'
OUTPUT_DIR = Project_Root / 'results' / 'detrending'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CASES = {name: {'metadata': Project_Root / 'data' / 'processed' / f'{name}_TESSCut_Metadata_Consolidated_QC.parquet', 'features': Project_Root / 'data' / 'processed' / f'{name}_TESSCut_features_QC.parquet', 'count': count} for name, count in [('SPOC', 506), ('QLP', 1355)]}
SPLIT_SEEDS = list(range(42, 142))
TEST_SIZE = 0.30
TARGET = 'family'
WorkerCount = 16
# Original detrending settings, unchanged.
gapThresholdDays = 1.0
minFinitePoints = 100
minSegmentPoints = 100
minSegmentDurationDays = 5.0
segmentDriftStrengthThreshold = 7.0
fractionSegmentsWithDriftThreshold = 0.5
detrendMethod = 'tesscut_segment_linear_mad_conditional_v1'
# Original feature extraction settings, unchanged.
MinPeriodDays = 0.05
MaxPeriodDays = 100.0
SamplesPerPeak = 10
MinCadences = 100
PhaseBinCount = 20
Eps = 1e-12
QualityScoreMap = {'clean': 0, 'acceptable': 1, 'caution': 2, 'poor': 3, 'missing': 4}
ProvenanceScoreMap = {'SPOC': 0, 'QLP': 1, 'TESSCut': 2}
FEATURE_RTOL = 1e-5
FEATURE_ATOL = 1e-7
CANONICAL_BASELINE = {'SPOC': (0.763158, 0.570680, 0.631579, 0.386845), 'QLP': (0.680590, 0.417110, 0.518428, 0.264590)}
rawPathColumn = "rawLightCurvePath"
provenanceColumn = "provenance"

from astropy import units as u
from astropy.time import Time
# Enable the copied standardizer; no acquisition stages are invoked.
RUN_LIGHT_CURVE_STANDARDIZATION = True


In [2]:
FEATURES = ['QualityScore', 'OriginalFluxMedian', 'OriginalFluxStd', 'CadenceCount', 'TimeSpanDays', 'MedianCadenceDays', 'FluxStd', 'FluxMedian', 'FluxMad', 'FluxP05', 'FluxP90', 'FluxP95', 'FluxIqr', 'FluxAmplitude', 'FluxPercentAmplitude95To5', 'FluxSkewness', 'FluxKurtosis', 'TailAsymmetry', 'TailRatio', 'EtaVonNeumann', 'MaxAbsSlope', 'MedianAbsSuccessiveDiff', 'FractionBeyond1Std', 'FractionBeyond2Std', 'LsBestPeriod', 'LsMaxPower', 'LsFalseAlarmProbability', 'LsPeriod2', 'LsPeriod3', 'LsPeakPowerGap12', 'LsPeakPowerGap23', 'LsPowerRatio21', 'LsPowerRatio31', 'LsPeriodRatio21', 'LsPeriodRatio31', 'PhaseCurveStd', 'PhasePeakPhase', 'PhaseTroughPhase', 'PhasePeakToTroughPhaseDelta']

assert len(FEATURES) == 39
assert "ProvenanceScore" not in FEATURES

print("Canonical model feature count:", len(FEATURES))
print("ProvenanceScore excluded:", "ProvenanceScore" not in FEATURES)
display(pd.DataFrame({"Feature": FEATURES}))

Canonical model feature count: 39
ProvenanceScore excluded: True


,Feature
0,QualityScore
1,OriginalFluxMedian
2,OriginalFluxStd
3,CadenceCount
4,TimeSpanDays
5,MedianCadenceDays
6,FluxStd
7,FluxMedian
8,FluxMad
9,FluxP05


## Pipeline detrending implementation

The detrending implementation reproduces the pipeline procedures and thresholds. Processing is restricted to matched TESSCut rows.


In [3]:
def resolve_raw_fits_path(path_value, fits_base: Path) -> Optional[Path]:
    """
    Resolve raw FITS path from metadata.

    Rules:
    1. If path is absolute and exists, use it.
    2. If path is relative to fits_base and exists, use fits_base / path.
    3. If only filename works under fits_base, use fits_base / basename.
    """
    if pd.isna(path_value):
        return None

    p = Path(str(path_value))

    if p.is_absolute() and p.exists():
        return p

    candidate = fits_base / p
    if candidate.exists():
        return candidate

    candidate = fits_base / p.name
    if candidate.exists():
        return candidate

    return None


def load_time_flux_from_raw_fits(raw_fits_path: Path) -> Tuple[np.ndarray, np.ndarray, fits.HDUList]:
    """
    Load TIME and FLUX from HDU[1] of a pipeline-generated FITS file.

    Returns time, flux, and the opened HDUList. Caller should close HDUList.
    """
    hdul = fits.open(raw_fits_path, memmap=False)

    if len(hdul) <= 1 or hdul[1].data is None:
        hdul.close()
        raise ValueError("Missing table data in HDU[1]")

    table = hdul[1].data
    names = set(getattr(table, "names", []) or [])

    if "TIME" not in names or "FLUX" not in names:
        hdul.close()
        raise ValueError("HDU[1] must contain TIME and FLUX columns")

    time = np.asarray(table["TIME"], dtype=float)
    flux = np.asarray(table["FLUX"], dtype=float)

    return time, flux, hdul


def split_into_segments(time: np.ndarray, flux: np.ndarray, gap_days: float) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Split sorted finite time/flux arrays into contiguous segments.

    Returns list of (segment_indices, segment_time, segment_flux).
    segment_indices refer to positions in the sorted finite arrays.
    """
    if len(time) == 0:
        return []

    dt = np.diff(time)
    breaks = np.where(dt > gap_days)[0] + 1
    starts = np.concatenate(([0], breaks))
    ends = np.concatenate((breaks, [len(time)]))

    segments = []
    for start, end in zip(starts, ends):
        if end > start:
            idx = np.arange(start, end)
            segments.append((idx, time[start:end], flux[start:end]))

    return segments


def analyze_segment_drift(
    segment_time: np.ndarray,
    segment_flux: np.ndarray,
    threshold: float,
    min_points: int,
    min_duration_days: float,
) -> Optional[Dict]:
    """
    Analyze linear drift in one contiguous segment.

    The metric compares total fitted linear drift across the segment
    to robust scatter estimated by 1.4826*MAD.
    """
    n = len(segment_time)
    if n < min_points:
        return None

    duration = float(segment_time[-1] - segment_time[0])
    if not np.isfinite(duration) or duration < min_duration_days:
        return None

    median_flux = float(np.median(segment_flux))
    mad = float(np.median(np.abs(segment_flux - median_flux)))
    robust_scatter = 1.4826 * mad

    if not np.isfinite(robust_scatter) or robust_scatter <= 0:
        return None

    t_centered = segment_time - np.median(segment_time)

    try:
        slope, intercept = np.polyfit(t_centered, segment_flux, 1)
    except Exception:
        return None

    drift = abs(float(slope) * duration)
    drift_strength = drift / robust_scatter

    if not np.isfinite(drift_strength):
        return None

    return {
        "segmentNumPoints": int(n),
        "segmentDurationDays": float(duration),
        "segmentSlope": float(slope),
        "segmentIntercept": float(intercept),
        "segmentDrift": float(drift),
        "segmentRobustScatter": float(robust_scatter),
        "segmentDriftStrength": float(drift_strength),
        "segmentHasDrift": bool(drift_strength >= threshold),
    }


def detect_robust_drift_for_file(raw_fits_path: Path) -> Dict:
    """
    Run segment-wise robust drift detection for one raw FITS file.
    """
    try:
        time, flux, hdul = load_time_flux_from_raw_fits(raw_fits_path)
        hdul.close()

        finite_mask = np.isfinite(time) & np.isfinite(flux)
        time_finite = time[finite_mask]
        flux_finite = flux[finite_mask]

        if len(time_finite) < minFinitePoints:
            return {
                "trendStatus": "insufficient_data",
                "trendDetected": False,
                "trendNumFinitePoints": int(len(time_finite)),
                "validSegmentCount": 0,
                "segmentDriftCount": 0,
                "fractionSegmentsWithDrift": np.nan,
                "maxSegmentDriftStrength": np.nan,
                "medianSegmentDriftStrength": np.nan,
            }

        sort_idx = np.argsort(time_finite)
        time_finite = time_finite[sort_idx]
        flux_finite = flux_finite[sort_idx]

        segments = split_into_segments(time_finite, flux_finite, gapThresholdDays)

        valid_results = []
        for _idx, seg_time, seg_flux in segments:
            result = analyze_segment_drift(
                segment_time=seg_time,
                segment_flux=seg_flux,
                threshold=segmentDriftStrengthThreshold,
                min_points=minSegmentPoints,
                min_duration_days=minSegmentDurationDays,
            )
            if result is not None:
                valid_results.append(result)

        if len(valid_results) == 0:
            return {
                "trendStatus": "insufficient_data",
                "trendDetected": False,
                "trendNumFinitePoints": int(len(time_finite)),
                "validSegmentCount": 0,
                "segmentDriftCount": 0,
                "fractionSegmentsWithDrift": np.nan,
                "maxSegmentDriftStrength": np.nan,
                "medianSegmentDriftStrength": np.nan,
            }

        strengths = np.array([r["segmentDriftStrength"] for r in valid_results], dtype=float)
        drift_count = int(sum(r["segmentHasDrift"] for r in valid_results))
        valid_count = len(valid_results)
        frac = drift_count / valid_count

        detected = bool(frac >= fractionSegmentsWithDriftThreshold)

        return {
            "trendStatus": "drift" if detected else "no_drift",
            "trendDetected": detected,
            "trendNumFinitePoints": int(len(time_finite)),
            "validSegmentCount": int(valid_count),
            "segmentDriftCount": int(drift_count),
            "fractionSegmentsWithDrift": float(frac),
            "maxSegmentDriftStrength": float(np.max(strengths)),
            "medianSegmentDriftStrength": float(np.median(strengths)),
        }

    except Exception as e:
        return {
            "trendStatus": "failed",
            "trendDetected": False,
            "trendError": str(e),
            "trendNumFinitePoints": 0,
            "validSegmentCount": 0,
            "segmentDriftCount": 0,
            "fractionSegmentsWithDrift": np.nan,
            "maxSegmentDriftStrength": np.nan,
            "medianSegmentDriftStrength": np.nan,
        }

In [4]:
def build_detrended_filename(raw_fits_path: Path) -> str:
    """
    Convert xxxx_raw.fits to xxxx_detrended.fits.
    """
    name = raw_fits_path.name
    if name.endswith("_raw.fits"):
        return name.replace("_raw.fits", "_detrended.fits")
    if name.endswith(".fits"):
        return name.replace(".fits", "_detrended.fits")
    return name + "_detrended.fits"


def detrend_raw_fits_file(raw_fits_path: Path, output_folder: Path) -> Dict:
    """
    Create a detrended FITS file using segment-wise linear detrending.

    Non-finite rows are preserved in position; only finite rows are used
    for fitting and corrected where possible.
    """
    hdul = None
    try:
        time, flux, hdul = load_time_flux_from_raw_fits(raw_fits_path)

        finite_mask = np.isfinite(time) & np.isfinite(flux)
        finite_positions = np.where(finite_mask)[0]

        if len(finite_positions) < minFinitePoints:
            hdul.close()
            return {
                "detrendStatus": "insufficient_data",
                "detrended": False,
                "detrendLightCurvePath": None,
                "detrendError": None,
                "detrendSegmentCount": 0,
                "detrendCorrectedSegmentCount": 0,
            }

        # Work in sorted finite space, then map back to original row positions.
        time_finite = time[finite_mask]
        flux_finite = flux[finite_mask]
        sort_idx = np.argsort(time_finite)

        time_sorted = time_finite[sort_idx]
        flux_sorted = flux_finite[sort_idx]
        original_positions_sorted = finite_positions[sort_idx]

        detrended_flux = np.array(flux, dtype=float, copy=True)

        segments = split_into_segments(time_sorted, flux_sorted, gapThresholdDays)

        corrected_segments = 0
        valid_segments = 0

        for seg_sorted_idx, seg_time, seg_flux in segments:
            analysis = analyze_segment_drift(
                segment_time=seg_time,
                segment_flux=seg_flux,
                threshold=segmentDriftStrengthThreshold,
                min_points=minSegmentPoints,
                min_duration_days=minSegmentDurationDays,
            )

            if analysis is None:
                continue

            valid_segments += 1

            # Detrend every valid segment for a file already selected for detrending.
            t_centered = seg_time - np.median(seg_time)
            slope, intercept = np.polyfit(t_centered, seg_flux, 1)
            model = slope * t_centered + intercept
            segment_median = np.median(seg_flux)
            corrected_flux = seg_flux - model + segment_median

            original_positions = original_positions_sorted[seg_sorted_idx]
            detrended_flux[original_positions] = corrected_flux
            corrected_segments += 1

        if corrected_segments == 0:
            hdul.close()
            return {
                "detrendStatus": "no_valid_segments",
                "detrended": False,
                "detrendLightCurvePath": None,
                "detrendError": None,
                "detrendSegmentCount": int(valid_segments),
                "detrendCorrectedSegmentCount": 0,
            }

        # Create output HDUList by copying original structure.
        new_hdul = fits.HDUList([hdu.copy() for hdu in hdul])
        hdul.close()

        table_hdu = new_hdul[1]
        table_data = table_hdu.data

        # Rebuild table with detrended FLUX and optional ORIGINAL_FLUX.
        rebuilt_cols = []
        existing_names = list(table_data.names)

        for col in table_hdu.columns:
            if col.name == "FLUX":
                rebuilt_cols.append(
                    fits.Column(
                        name="FLUX",
                        array=np.asarray(detrended_flux, dtype=np.float32),
                        format=col.format,
                    )
                )
            else:
                rebuilt_cols.append(
                    fits.Column(
                        name=col.name,
                        array=table_data[col.name],
                        format=col.format,
                    )
                )

        if "ORIGINAL_FLUX" not in existing_names:
            rebuilt_cols.append(
                fits.Column(
                    name="ORIGINAL_FLUX",
                    array=np.asarray(flux, dtype=np.float32),
                    format="E",
                )
            )

        new_table_hdu = fits.BinTableHDU.from_columns(rebuilt_cols)
        new_table_hdu.header.extend(table_hdu.header, update=True, strip=True)

        # Record detrending metadata.
        new_table_hdu.header["DETRND"] = (True, "Segment-wise linear detrending applied")
        new_table_hdu.header["DTRMETH"] = (detrendMethod, "Detrending method")
        new_table_hdu.header["DTRTHR"] = (float(segmentDriftStrengthThreshold), "Drift-strength threshold")
        new_table_hdu.header["DTRFRAC"] = (float(fractionSegmentsWithDriftThreshold), "Segment fraction threshold")
        new_table_hdu.header["GAPDAYS"] = (float(gapThresholdDays), "Gap threshold in days")

        new_hdul[1] = new_table_hdu

        output_name = build_detrended_filename(raw_fits_path)
        output_path = output_folder / output_name
        new_hdul.writeto(output_path, overwrite=True)
        new_hdul.close()

        return {
            "detrendStatus": "detrended",
            "detrended": True,
            "detrendLightCurvePath": str(output_path),
            "detrendError": None,
            "detrendSegmentCount": int(valid_segments),
            "detrendCorrectedSegmentCount": int(corrected_segments),
        }

    except Exception as e:
        if hdul is not None:
            try:
                hdul.close()
            except Exception:
                pass

        return {
            "detrendStatus": "failed",
            "detrended": False,
            "detrendLightCurvePath": None,
            "detrendError": str(e),
            "detrendSegmentCount": 0,
            "detrendCorrectedSegmentCount": 0,
        }

def conditional_detrend_comparison_tesscut(
    input_metadata_file,
    output_metadata_file,
    output_folder,
):
    """
    Reuse the Chapter 5 drift-detection and conditional-detrending functions
    for one Chapter 6 TESSCut comparison metadata file.
    """
    input_path = Path(input_metadata_file)
    output_path = Path(output_metadata_file)
    output_folder = Path(output_folder)

    if not input_path.exists():
        raise FileNotFoundError(f"Input metadata parquet does not exist: {input_path}")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_folder.mkdir(parents=True, exist_ok=True)

    comparison_df = pd.read_parquet(input_path).copy()

    if rawPathColumn not in comparison_df.columns:
        raise ValueError(f"Required column missing: {rawPathColumn}")

    if "fitsQcStatus" not in comparison_df.columns:
        raise ValueError(
            "Required QC column missing: fitsQcStatus. Run Section 6.3 first."
        )

    new_columns_defaults = {
        "trendDetectionEligible": False,
        "trendDetectedForDetrend": False,
        "trendDetectionStatus": None,
        "trendDetectionMethod": "segment_wise_linear_drift_mad_v1",
        "trendDetectionThreshold": segmentDriftStrengthThreshold,
        "trendFractionSegmentsWithDrift": np.nan,
        "trendValidSegmentCount": 0,
        "trendSegmentDriftCount": 0,
        "trendMaxSegmentDriftStrength": np.nan,
        "trendMedianSegmentDriftStrength": np.nan,
        "trendNumFinitePoints": 0,
        "detrended": False,
        "detrendEligible": True,
        "detrendStatus": "not_processed",
        "detrendReason": None,
        "detrendMethod": None,
        "detrendLightCurvePath": None,
        "detrendInputRawLightCurvePath": None,
        "detrendResolvedRawFitsPath": None,
        "detrendSegmentCount": 0,
        "detrendCorrectedSegmentCount": 0,
        "detrendError": None,
    }

    for column, default in new_columns_defaults.items():
        if column not in comparison_df.columns:
            comparison_df[column] = default

    processed = 0
    missing_raw = 0
    detected_count = 0
    detrended_count = 0
    failed_count = 0

    for idx, row in comparison_df.iterrows():
        qc_status = str(row.get("fitsQcStatus", "unknown")).lower()
        if qc_status not in {"pass", "warning"}:
            comparison_df.at[idx, "trendDetectionEligible"] = False
            comparison_df.at[idx, "detrendEligible"] = False
            comparison_df.at[idx, "trendDetectedForDetrend"] = False
            comparison_df.at[idx, "detrended"] = False
            comparison_df.at[idx, "detrendStatus"] = "not_eligible_qc"
            comparison_df.at[idx, "detrendReason"] = (
                f"fitsQcStatus={qc_status}; only pass/warning rows are eligible"
            )
            continue

        comparison_df.at[idx, "trendDetectionEligible"] = True
        comparison_df.at[idx, "detrendEligible"] = True
        comparison_df.at[idx, "detrendInputRawLightCurvePath"] = row.get(rawPathColumn)

        raw_fits_path = resolve_raw_fits_path(
            row.get(rawPathColumn),
            output_folder,
        )

        if raw_fits_path is None:
            missing_raw += 1
            comparison_df.at[idx, "trendDetectionStatus"] = "missing_raw_fits"
            comparison_df.at[idx, "trendDetectedForDetrend"] = False
            comparison_df.at[idx, "detrended"] = False
            comparison_df.at[idx, "detrendStatus"] = "missing_raw_fits"
            comparison_df.at[idx, "detrendReason"] = "raw FITS path could not be resolved"
            continue

        comparison_df.at[idx, "detrendResolvedRawFitsPath"] = str(raw_fits_path)

        trend = detect_robust_drift_for_file(raw_fits_path)

        comparison_df.at[idx, "trendDetectionStatus"] = trend.get("trendStatus")
        comparison_df.at[idx, "trendDetectedForDetrend"] = bool(
            trend.get("trendDetected", False)
        )
        comparison_df.at[idx, "trendFractionSegmentsWithDrift"] = trend.get(
            "fractionSegmentsWithDrift", np.nan
        )
        comparison_df.at[idx, "trendValidSegmentCount"] = trend.get(
            "validSegmentCount", 0
        )
        comparison_df.at[idx, "trendSegmentDriftCount"] = trend.get(
            "segmentDriftCount", 0
        )
        comparison_df.at[idx, "trendMaxSegmentDriftStrength"] = trend.get(
            "maxSegmentDriftStrength", np.nan
        )
        comparison_df.at[idx, "trendMedianSegmentDriftStrength"] = trend.get(
            "medianSegmentDriftStrength", np.nan
        )
        comparison_df.at[idx, "trendNumFinitePoints"] = trend.get(
            "trendNumFinitePoints", 0
        )

        if "trendError" in trend:
            comparison_df.at[idx, "detrendError"] = trend["trendError"]

        if not bool(trend.get("trendDetected", False)):
            comparison_df.at[idx, "detrended"] = False
            comparison_df.at[idx, "detrendStatus"] = "not_detrended_no_drift"
            comparison_df.at[idx, "detrendReason"] = (
                "TESSCut but robust drift not detected"
            )
            processed += 1
            continue

        detected_count += 1

        detrend_result = detrend_raw_fits_file(raw_fits_path, output_folder)

        for key, value in detrend_result.items():
            if key in comparison_df.columns:
                comparison_df.at[idx, key] = value

        comparison_df.at[idx, "detrendMethod"] = detrendMethod
        comparison_df.at[idx, "detrendReason"] = (
            "TESSCut and robust drift detected"
        )

        if detrend_result.get("detrended", False):
            detrended_count += 1
        elif detrend_result.get("detrendStatus") == "failed":
            failed_count += 1

        processed += 1

        if processed % 250 == 0:
            print(
                f"{input_path.name}: processed={processed}, "
                f"trend_detected={detected_count}, "
                f"detrended={detrended_count}, "
                f"missing_raw={missing_raw}, failed={failed_count}"
            )

    comparison_df.to_parquet(output_path, index=False)

    summary = {
        "input": str(input_path),
        "output": str(output_path),
        "rows": int(len(comparison_df)),
        "processed": int(processed),
        "trendDetected": int(detected_count),
        "detrended": int(detrended_count),
        "missingRaw": int(missing_raw),
        "failed": int(failed_count),
    }

    print(summary)
    return comparison_df, summary


## Consolidated feature extractor

The feature-extraction implementation reproduces the consolidated extractor settings and feature definitions. Model evaluation uses the common 39-predictor feature set.


In [5]:
def SafeFloat(Value: Any) -> float:
    try:
        FloatValue = float(Value)
    except Exception:
        return np.nan
    return FloatValue if np.isfinite(FloatValue) else np.nan


def SafeBool(Value: Any) -> bool:
    if Value is None:
        return False
    if isinstance(Value, float) and pd.isna(Value):
        return False
    return bool(Value)


def SafeDivide(Numerator: float, Denominator: float, Eps: float = Eps) -> float:
    if not np.isfinite(Numerator) or not np.isfinite(Denominator) or abs(Denominator) < Eps:
        return np.nan
    return float(Numerator / Denominator)


def ResolvePath(PathValue: Any, MetadataPath: Path) -> Optional[Path]:
    if PathValue is None:
        return None
    if isinstance(PathValue, float) and pd.isna(PathValue):
        return None

    PathObj = Path(str(PathValue))
    if PathObj.is_absolute() and PathObj.exists():
        return PathObj
    if PathObj.exists():
        return PathObj

    CandidatePath = MetadataPath.parent / PathObj
    if CandidatePath.exists():
        return CandidatePath

    return PathObj

In [6]:
def LoadLightCurve(LightCurvePath: Path) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """Load a light curve while treating flux uncertainty as optional.

    Time and flux determine whether a cadence is scientifically usable.
    FluxErr is retained only when it is aligned with the loaded arrays and all
    retained uncertainty values are finite and strictly positive. Invalid or
    missing uncertainty therefore downgrades Lomb-Scargle from weighted to
    unweighted analysis instead of removing otherwise valid QLP cadences.
    """

    def FirstAvailableColumn(ColumnNames: list[str], CandidateNames: list[str]) -> Optional[str]:
        NameMap = {Name.upper(): Name for Name in ColumnNames}
        for CandidateName in CandidateNames:
            MatchedName = NameMap.get(CandidateName.upper())
            if MatchedName is not None:
                return MatchedName
        return None

    def ReadGenericFitsTable(FitsPath: Path) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
        with fits.open(str(FitsPath), memmap=False) as Hdul:
            TableData = None

            for Hdu in Hdul:
                Data = getattr(Hdu, "data", None)
                if Data is not None and getattr(Data, "dtype", None) is not None and Data.dtype.names:
                    TableData = Data
                    break

            if TableData is None:
                raise ValueError("No FITS binary table with named columns was found")

            ColumnNames = list(TableData.dtype.names)

            TimeColumn = FirstAvailableColumn(
                ColumnNames,
                ["TIME", "TMID", "BJD"],
            )
            FluxColumn = FirstAvailableColumn(
                ColumnNames,
                ["FLUX", "SAP_FLUX", "PDCSAP_FLUX", "KSPSAP_FLUX", "DET_FLUX", "NORM_FLUX"],
            )
            FluxErrColumn = FirstAvailableColumn(
                ColumnNames,
                ["FLUX_ERR", "SAP_FLUX_ERR", "PDCSAP_FLUX_ERR", "KSPSAP_FLUX_ERR", "ERR_FLUX"],
            )

            if TimeColumn is None or FluxColumn is None:
                raise ValueError(
                    f"Required time/flux columns are missing. Available columns: {ColumnNames}"
                )

            Time = np.asarray(TableData[TimeColumn], dtype=float).reshape(-1)
            Flux = np.asarray(TableData[FluxColumn], dtype=float).reshape(-1)
            FluxErr = None

            if FluxErrColumn is not None:
                FluxErr = np.asarray(TableData[FluxErrColumn], dtype=float).reshape(-1)

            return Time, Flux, FluxErr

    def ExtractFromLightkurve(LightCurve) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
        Time = np.asarray(
            getattr(LightCurve.time, "value", LightCurve.time),
            dtype=float,
        ).reshape(-1)

        Flux = np.asarray(
            getattr(LightCurve.flux, "value", LightCurve.flux),
            dtype=float,
        ).reshape(-1)

        FluxErr = None
        if hasattr(LightCurve, "flux_err") and LightCurve.flux_err is not None:
            FluxErr = np.asarray(
                getattr(LightCurve.flux_err, "value", LightCurve.flux_err),
                dtype=float,
            ).reshape(-1)

        return Time, Flux, FluxErr

    LightCurvePath = Path(LightCurvePath)

    if not LightCurvePath.exists():
        raise FileNotFoundError(str(LightCurvePath))

    Time = None
    Flux = None
    FluxErr = None
    ReadErrors = []

    # Attempt 1: standard lightkurve reader.
    try:
        LightCurve = lk.read(str(LightCurvePath))
        Time, Flux, FluxErr = ExtractFromLightkurve(LightCurve)
    except Exception as Exc:
        ReadErrors.append(f"generic lk.read: {Exc!r}")

    # Attempt 2: QLP-specific reader.
    if Time is None or Flux is None:
        try:
            LightCurve = lk.read(
                str(LightCurvePath),
                author="QLP",
                flux_column="kspsap_flux",
            )
            Time, Flux, FluxErr = ExtractFromLightkurve(LightCurve)
        except Exception as Exc:
            ReadErrors.append(f"QLP lk.read: {Exc!r}")

    # Attempt 3: direct FITS table parsing.
    if Time is None or Flux is None:
        try:
            Time, Flux, FluxErr = ReadGenericFitsTable(LightCurvePath)
        except Exception as Exc:
            ReadErrors.append(f"generic FITS parsing: {Exc!r}")
            raise ValueError(
                "Unable to read light curve. " + " | ".join(ReadErrors)
            ) from Exc

    if Time.shape != Flux.shape:
        raise ValueError(
            f"Time/Flux shape mismatch: Time={Time.shape}, Flux={Flux.shape}"
        )

    # Only Time and Flux define a valid cadence.
    CadenceMask = np.isfinite(Time) & np.isfinite(Flux)

    FilteredTime = Time[CadenceMask]
    FilteredFlux = Flux[CadenceMask]
    FilteredFluxErr = None

    # FluxErr is optional. It must never remove otherwise valid Time/Flux rows.
    if FluxErr is not None:
        if FluxErr.shape == Time.shape:
            CandidateFluxErr = FluxErr[CadenceMask]
            ValidFluxErr = np.isfinite(CandidateFluxErr) & (CandidateFluxErr > 0)

            if len(CandidateFluxErr) > 0 and np.all(ValidFluxErr):
                FilteredFluxErr = CandidateFluxErr
            else:
                Logger.debug(
                    "Ignoring unusable flux_err for %s: %d/%d retained values are finite and positive",
                    LightCurvePath,
                    int(np.sum(ValidFluxErr)),
                    len(CandidateFluxErr),
                )
        else:
            Logger.debug(
                "Ignoring flux_err shape mismatch for %s: flux_err=%s, time=%s",
                LightCurvePath,
                FluxErr.shape,
                Time.shape,
            )

    # Keep samples in chronological order.
    Order = np.argsort(FilteredTime)
    FilteredTime = FilteredTime[Order]
    FilteredFlux = FilteredFlux[Order]

    if FilteredFluxErr is not None:
        FilteredFluxErr = FilteredFluxErr[Order]

    return FilteredTime, FilteredFlux, FilteredFluxErr

In [7]:
def ExtractIdentifierAndMetadataFeatures(Row: pd.Series) -> Dict[str, Any]:
    Result: Dict[str, Any] = {}

    ColumnsToPreserve = [
        "family", "VSXType", "VSXId", "Name", "ticId", "bestTicId",
        "ticDistanceArcmin", "lightCurvePath", "rawLightCurvePath",
        "trendFlag", "trendScore", "adfPValue",
    ]

    for ColumnName in ColumnsToPreserve:
        if ColumnName in Row.index:
            Result[ColumnName] = Row.get(ColumnName)

    QualityValue = Row.get("quality", Row.get("fitsQcStatus", Row.get("lightCurveQuality", np.nan)))
    ProvenanceValue = Row.get("provenance", Row.get("author", np.nan))

    QualityLabel = str(QualityValue) if not pd.isna(QualityValue) else "missing"
    ProvenanceLabel = str(ProvenanceValue) if not pd.isna(ProvenanceValue) else "missing"

    Result.update({
        "QualityLabel": QualityLabel,
        "QualityScore": QualityScoreMap.get(QualityLabel, np.nan),
        "Provenance": ProvenanceLabel,
        "ProvenanceScore": ProvenanceScoreMap.get(ProvenanceLabel, np.nan),
        "OriginalFluxMedian": SafeFloat(Row.get("fluxMedian", np.nan)),
        "OriginalFluxStd": SafeFloat(Row.get("fluxStd", np.nan)),
        "OriginalFluxSnr": SafeFloat(Row.get("fluxSnr", np.nan)),
        "LowSnr": SafeBool(Row.get("lowSNR", False)),
        "LowQualityLightCurve": SafeBool(Row.get("lowQualityLightCurve", False)),
    })
    return Result

In [8]:
def ExtractBasicStatisticalFeatures(Time: np.ndarray, Flux: np.ndarray) -> Dict[str, float]:
    CadenceCount = len(Flux)
    P01, P05, P10, P25, P50, P75, P90, P95, P99 = np.percentile(
        Flux, [1, 5, 10, 25, 50, 75, 90, 95, 99]
    )
    FluxStd = float(np.std(Flux))

    return {
        "CadenceCount": float(CadenceCount),
        "TimeSpanDays": float(np.max(Time) - np.min(Time)) if CadenceCount > 1 else np.nan,
        "MedianCadenceDays": float(np.median(np.diff(Time))) if CadenceCount > 2 else np.nan,
        "FluxStd": FluxStd,
        "FluxMedian": float(P50),
        "FluxMad": float(np.median(np.abs(Flux - P50))),
        "FluxP05": float(P05),
        "FluxP10": float(P10),
        "FluxP90": float(P90),
        "FluxP95": float(P95),
        "FluxIqr": float(P75 - P25),
        "FluxAmplitude": float(np.max(Flux) - np.min(Flux)),
        "FluxPercentAmplitude95To5": float(P95 - P05),
        "FluxPercentAmplitude90To10": float(P90 - P10),
        "FluxSkewness": float(skew(Flux, bias=False)) if CadenceCount >= 3 and FluxStd > Eps else np.nan,
        "FluxKurtosis": float(kurtosis(Flux, bias=False)) if CadenceCount >= 4 and FluxStd > Eps else np.nan,
    }

In [9]:
def ExtractTailAsymmetryFeatures(Flux: np.ndarray) -> Dict[str, float]:
    P05, P50, P95 = np.percentile(Flux, [5, 50, 95])

    TailUpper = float(P95 - P50)
    TailLower = float(P50 - P05)
    TailAsymmetry = float(TailUpper - TailLower)
    TailRatio = SafeDivide(TailUpper, TailLower + Eps)

    return {
        "TailUpper": TailUpper,
        "TailLower": TailLower,
        "TailAsymmetry": TailAsymmetry,
        "TailRatio": TailRatio,
    }

In [10]:
def ExtractVariabilityFeatures(Time: np.ndarray, Flux: np.ndarray) -> Dict[str, float]:
    CadenceCount = len(Flux)
    if CadenceCount < 3:
        return {
            "EtaVonNeumann": np.nan,
            "MaxAbsSlope": np.nan,
            "MedianAbsSuccessiveDiff": np.nan,
            "FractionBeyond1Std": np.nan,
            "FractionBeyond2Std": np.nan,
        }

    FluxMean = float(np.mean(Flux))
    FluxStd = float(np.std(Flux))
    FluxVariance = float(np.var(Flux))
    FluxDiff = np.diff(Flux)
    TimeDiff = np.diff(Time)

    ValidTimeDiff = np.isfinite(TimeDiff) & (np.abs(TimeDiff) > Eps)
    Slopes = FluxDiff[ValidTimeDiff] / TimeDiff[ValidTimeDiff] if np.any(ValidTimeDiff) else np.array([])

    Eta = np.sum(FluxDiff ** 2) / ((CadenceCount - 1) * FluxVariance) if FluxVariance > Eps else np.nan

    return {
        "EtaVonNeumann": float(Eta) if np.isfinite(Eta) else np.nan,
        "MaxAbsSlope": float(np.max(np.abs(Slopes))) if Slopes.size else np.nan,
        "MedianAbsSuccessiveDiff": float(np.median(np.abs(FluxDiff))),
        "FractionBeyond1Std": float(np.mean(np.abs(Flux - FluxMean) > FluxStd)) if FluxStd > Eps else np.nan,
        "FractionBeyond2Std": float(np.mean(np.abs(Flux - FluxMean) > 2 * FluxStd)) if FluxStd > Eps else np.nan,
    }

In [11]:
def ExtractLombScargleFeatures(Time: np.ndarray, Flux: np.ndarray, FluxErr: Optional[np.ndarray]) -> Dict[str, float]:
    Result = {
        "LsBestPeriod": np.nan,
        "LsMaxPower": np.nan,
        "LsFalseAlarmProbability": np.nan,
        "LsPeriod2": np.nan,
        "LsPeriod3": np.nan,
        "LsPower2": np.nan,
        "LsPower3": np.nan,
        "LsPeakPowerGap12": np.nan,
        "LsPeakPowerGap23": np.nan,
        "LsPowerRatio21": np.nan,
        "LsPowerRatio31": np.nan,
        "LsPeriodRatio21": np.nan,
        "LsPeriodRatio31": np.nan,
    }

    if len(Flux) < MinCadences:
        return Result

    TimeSpan = float(np.max(Time) - np.min(Time))
    if not np.isfinite(TimeSpan) or TimeSpan <= 0:
        return Result

    MaxPeriod = min(MaxPeriodDays, 0.9 * TimeSpan)
    if MaxPeriod <= MinPeriodDays:
        return Result

    CenteredFlux = Flux - np.nanmedian(Flux)

    try:
        if FluxErr is not None and len(FluxErr) == len(Flux) and np.all(np.isfinite(FluxErr)):
            LombScargleModel = LombScargle(Time, CenteredFlux, dy=FluxErr)
        else:
            LombScargleModel = LombScargle(Time, CenteredFlux)

        Frequency, Power = LombScargleModel.autopower(
            minimum_frequency=1.0 / MaxPeriod,
            maximum_frequency=1.0 / MinPeriodDays,
            samples_per_peak=SamplesPerPeak,
        )

        FiniteMask = np.isfinite(Frequency) & np.isfinite(Power) & (Frequency > 0)
        Frequency = Frequency[FiniteMask]
        Power = Power[FiniteMask]
        if len(Power) == 0:
            return Result

        Order = np.argsort(Power)[::-1]
        BestFrequency = float(Frequency[Order[0]])
        BestPeriod = float(1.0 / BestFrequency)
        BestPower = float(Power[Order[0]])

        Result.update({
            "LsBestPeriod": BestPeriod,
            "LsMaxPower": BestPower,
            "LsFalseAlarmProbability": float(LombScargleModel.false_alarm_probability(BestPower)),
        })

        if len(Order) > 1:
            Period2 = float(1.0 / Frequency[Order[1]])
            Power2 = float(Power[Order[1]])
            Result.update({
                "LsPeriod2": Period2,
                "LsPower2": Power2,
                "LsPowerRatio21": SafeDivide(Power2, BestPower),
                "LsPeriodRatio21": SafeDivide(Period2, BestPeriod),
            })
            Result["LsPeakPowerGap12"] = BestPower - Power2

        if len(Order) > 2:
            Period3 = float(1.0 / Frequency[Order[2]])
            Power3 = float(Power[Order[2]])
            Result.update({
                "LsPeriod3": Period3,
                "LsPower3": Power3,
                "LsPowerRatio31": SafeDivide(Power3, BestPower),
                "LsPeriodRatio31": SafeDivide(Period3, BestPeriod),
            })
            if np.isfinite(Result.get("LsPower2", np.nan)):
                Result["LsPeakPowerGap23"] = Result["LsPower2"] - Power3

    except Exception as Exc:
        Logger.debug("Lomb-Scargle failed: %s", Exc)

    return Result

In [12]:
def ExtractPhaseFeatures(Time: np.ndarray, Flux: np.ndarray, Period: float) -> Dict[str, float]:
    Result = {
        "PhaseCurveStd": np.nan,
        "PhaseCurveRange": np.nan,
        "PhaseCurveSmoothness": np.nan,
        "PhasePeakPhase": np.nan,
        "PhaseTroughPhase": np.nan,
        "PhasePeakToTroughPhaseDelta": np.nan,
    }

    if not np.isfinite(Period) or Period <= 0 or len(Flux) < MinCadences:
        return Result

    try:
        Phase = (Time % Period) / Period
        Order = np.argsort(Phase)
        Phase = Phase[Order]
        Flux = Flux[Order]

        BinEdges = np.linspace(0.0, 1.0, PhaseBinCount + 1)
        BinIndex = np.digitize(Phase, BinEdges) - 1

        BinnedPhase = []
        BinnedFlux = []

        for BinNumber in range(PhaseBinCount):
            BinMask = BinIndex == BinNumber
            if np.any(BinMask):
                BinnedPhase.append(float(np.median(Phase[BinMask])))
                BinnedFlux.append(float(np.median(Flux[BinMask])))

        BinnedPhase = np.asarray(BinnedPhase, dtype=float)
        BinnedFlux = np.asarray(BinnedFlux, dtype=float)
        if len(BinnedFlux) < 5:
            return Result

        PeakIndex = int(np.argmax(BinnedFlux))
        TroughIndex = int(np.argmin(BinnedFlux))
        PeakPhase = float(BinnedPhase[PeakIndex])
        TroughPhase = float(BinnedPhase[TroughIndex])

        RawDelta = abs(PeakPhase - TroughPhase)
        CyclicDelta = min(RawDelta, 1.0 - RawDelta)
        CyclicDiff = np.diff(np.r_[BinnedFlux, BinnedFlux[0]])

        Result.update({
            "PhaseCurveStd": float(np.std(BinnedFlux)),
            "PhaseCurveRange": float(np.max(BinnedFlux) - np.min(BinnedFlux)),
            "PhaseCurveSmoothness": float(np.std(CyclicDiff)),
            "PhasePeakPhase": PeakPhase,
            "PhaseTroughPhase": TroughPhase,
            "PhasePeakToTroughPhaseDelta": float(CyclicDelta),
        })

    except Exception as Exc:
        Logger.debug("Phase feature extraction failed: %s", Exc)

    return Result

In [13]:
def provenance_column(df):
    for key in ['Provenance', 'provenance', 'Source', 'source', 'author']:
        if key in df.columns: return key
    raise KeyError('Missing provenance column')

def normalized(value):
    text = str(value).strip().upper()
    return 'TESSCut' if text in {'TESSCUT', 'TESS CUT'} else text

def indexed(df, provenance):
    df = df.copy()
    df['VSXId'] = df['VSXId'].astype(str)
    df = df[df[provenance_column(df)].map(normalized).eq(provenance)]
    counts = df.groupby('VSXId').size()
    return df[df['VSXId'].isin(counts.index[counts.eq(1)])].set_index('VSXId', drop=False).sort_index()

def resolve_source(value, cache_root):
    if value is None or pd.isna(value): raise ValueError('Missing rawLightCurvePath')
    p = Path(str(value))
    candidates = [p] if p.is_absolute() else [RawDataRoot / p, cache_root / p, cache_root / p.name]
    for candidate in candidates:
        if candidate.is_file(): return candidate.resolve()
    raise FileNotFoundError(str(value))

def ExtractFeaturesForRow(Row: pd.Series, MetadataPath: Path) -> Dict[str, Any]:
    fitsBasePath = RawDataRoot
    Result: Dict[str, Any] = {
        "FeatureStatus": "unknown",
        "FeatureError": None,
    }

    Result.update(ExtractIdentifierAndMetadataFeatures(Row))
    # LightCurvePath = ResolvePath(Row.get("lightCurvePath"), MetadataPath)
    LightCurvePath = Path(fitsBasePath / Row.get("lightCurvePath"))

    if LightCurvePath is None:
        Result["FeatureStatus"] = "missing_lightcurve_path"
        Result["FeatureError"] = "No lightCurvePath"
        return Result

    if not LightCurvePath.exists():
        Result["FeatureStatus"] = "missing_lightcurve_file"
        Result["FeatureError"] = str(LightCurvePath)
        return Result

    try:
        Time, Flux, FluxErr = LoadLightCurve(LightCurvePath)

        if len(Flux) < MinCadences:
            Result["FeatureStatus"] = "too_few_cadences"
            Result["FeatureError"] = f"Only {len(Flux)} finite cadences"
            Result["CadenceCount"] = float(len(Flux))
            return Result

        Result.update(ExtractBasicStatisticalFeatures(Time, Flux))
        Result.update(ExtractTailAsymmetryFeatures(Flux))
        Result.update(ExtractVariabilityFeatures(Time, Flux))

        LombScargleFeatures = ExtractLombScargleFeatures(Time, Flux, FluxErr)
        Result.update(LombScargleFeatures)

        BestPeriod = LombScargleFeatures.get("LsBestPeriod", np.nan)
        Result.update(ExtractPhaseFeatures(Time, Flux, BestPeriod))

        Result["FeatureStatus"] = "ok"

    except Exception as Exc:
        Result["FeatureStatus"] = "failed"
        Result["FeatureError"] = repr(Exc)

    return Result

def FinalizeFeatureDf(FeatureDf: pd.DataFrame) -> pd.DataFrame:
    if "_InputOrder" in FeatureDf.columns:
        FeatureDf = FeatureDf.sort_values("_InputOrder").drop(columns=["_InputOrder"]).reset_index(drop=True)
    return FeatureDf


def ExtractFeaturesSerial(MetadataDf: pd.DataFrame, MetadataPath: Path) -> pd.DataFrame:
    ResultRows = []
    for Position, (_, Row) in enumerate(MetadataDf.iterrows()):
        if Position % 100 == 0:
            Logger.info("Processing %s/%s", Position, len(MetadataDf))
        FeatureRow = ExtractFeaturesForRow(Row, MetadataPath)
        FeatureRow["_InputOrder"] = Position
        ResultRows.append(FeatureRow)
    return FinalizeFeatureDf(pd.DataFrame(ResultRows))


def ExtractFeaturesParallel(MetadataDf: pd.DataFrame, MetadataPath: Path, WorkerCount: int = WorkerCount) -> pd.DataFrame:
    ResultRows = []
    TotalRows = len(MetadataDf)
    Logger.info("Starting parallel feature extraction with WorkerCount=%s", WorkerCount)

    with ThreadPoolExecutor(max_workers=WorkerCount) as Executor:
        FutureMap = {}
        for Position, (_, Row) in enumerate(MetadataDf.iterrows()):
            Future = Executor.submit(ExtractFeaturesForRow, Row, MetadataPath)
            FutureMap[Future] = Position

        CompletedCount = 0
        for Future in as_completed(FutureMap):
            Position = FutureMap[Future]
            try:
                FeatureRow = Future.result()
            except Exception as Exc:
                FeatureRow = {"FeatureStatus": "failed", "FeatureError": repr(Exc)}

            FeatureRow["_InputOrder"] = Position
            ResultRows.append(FeatureRow)
            CompletedCount += 1

            if CompletedCount % 100 == 0 or CompletedCount == TotalRows:
                Logger.info("Completed %s/%s", CompletedCount, TotalRows)

    return FinalizeFeatureDf(pd.DataFrame(ResultRows))


def ExtractFeatures(MetadataDf: pd.DataFrame, MetadataPath: Path, WorkerCount: int = WorkerCount) -> pd.DataFrame:
    if WorkerCount <= 1:
        return ExtractFeaturesSerial(MetadataDf, MetadataPath)
    return ExtractFeaturesParallel(MetadataDf, MetadataPath, WorkerCount=WorkerCount)

matched = {}
for name, case in CASES.items():
    old = pd.read_parquet(case['features'])
    official, tc = indexed(old, name), indexed(old, 'TESSCut')
    ids = official.index.intersection(tc.index).sort_values()
    counts = official.loc[ids, TARGET].astype(str).value_counts()
    ids = ids[official.loc[ids, TARGET].astype(str).isin(counts[counts >= 2].index)]
    if len(ids) != case['count']: raise ValueError(f'{name}: expected {case["count"]} matched stars, found {len(ids)}')
    official, tc = official.loc[ids], tc.loc[ids]
    if not official[TARGET].astype(str).equals(tc[TARGET].astype(str)): raise ValueError('Feature labels disagree')
    if set(FEATURES) - set(old): raise ValueError('Canonical 39-feature schema missing')
    metadata = indexed(pd.read_parquet(case['metadata']), 'TESSCut')
    if not ids.isin(metadata.index).all(): raise ValueError('Matched TESSCut metadata missing or duplicated')
    metadata = metadata.loc[ids].copy()
    if not metadata[TARGET].astype(str).equals(tc[TARGET].astype(str)): raise ValueError('Metadata labels disagree')
    metadata['provenance'] = 'TESSCut'
    if 'fitsQcStatus' not in metadata: raise ValueError('Missing fitsQcStatus; use QC metadata')
    matched[name] = dict(official=official, original=tc, metadata=metadata)
    print(name, 'matched stars:', len(ids))

class MatchedLightCurveStandardizer:
    def __init__(self):
        self.logger = Logger
        self.normalizationStdFloor = 1e-8
        self.lowSnrThreshold = 3.0

    def _readStoredRawLightCurve(self, rawPath, provenance):
            """
            Read a raw light curve saved by this pipeline.

            Lightkurve is attempted first. A generic FITS-table reader handles
            QLP, TESSCut, or fallback files that Lightkurve cannot identify.
            """
            try:
                return lk.read(rawPath)
            except Exception as lkExc:
                self.logger.warning(
                    "lk.read failed for %s provenance=%s; using generic FITS reader: %s",
                    rawPath,
                    provenance,
                    lkExc,
                )

            with fits.open(rawPath, memmap=False) as hdul:
                tableHdu = next(
                    (
                        hdu
                        for hdu in hdul
                        if isinstance(hdu, fits.BinTableHDU)
                        and hdu.data is not None
                    ),
                    None,
                )
                if tableHdu is None:
                    raise ValueError(f"No FITS binary table found in {rawPath}")

                columnNames = {
                    name.upper(): name
                    for name in tableHdu.columns.names
                }

                timeColumn = columnNames.get("TIME")
                fluxColumn = (
                    columnNames.get("FLUX")
                    or columnNames.get("PDCSAP_FLUX")
                    or columnNames.get("SAP_FLUX")
                )
                fluxErrColumn = (
                    columnNames.get("FLUX_ERR")
                    or columnNames.get("PDCSAP_FLUX_ERR")
                    or columnNames.get("SAP_FLUX_ERR")
                )
                qualityColumn = columnNames.get("QUALITY")

                if timeColumn is None or fluxColumn is None:
                    raise ValueError(
                        f"Required TIME/FLUX columns not found in {rawPath}; "
                        f"columns={tableHdu.columns.names}"
                    )

                timeValues = np.asarray(tableHdu.data[timeColumn], dtype=float)
                fluxValues = np.asarray(tableHdu.data[fluxColumn], dtype=float)
                valid = np.isfinite(timeValues) & np.isfinite(fluxValues)

                lightCurveArgs = {
                    "time": Time(timeValues[valid], format="jd"),
                    "flux": fluxValues[valid] * u.dimensionless_unscaled,
                }

                if fluxErrColumn is not None:
                    fluxErrValues = np.asarray(
                        tableHdu.data[fluxErrColumn],
                        dtype=float,
                    )
                    lightCurveArgs["flux_err"] = (
                        fluxErrValues[valid] * u.dimensionless_unscaled
                    )

                lightCurve = lk.LightCurve(**lightCurveArgs)

                if qualityColumn is not None:
                    qualityValues = np.asarray(tableHdu.data[qualityColumn])
                    lightCurve["quality"] = qualityValues[valid]

                return lightCurve

    def _standardizeLightCurve(self, lightCurve):
            if not RUN_LIGHT_CURVE_STANDARDIZATION:
                return lightCurve, None
        
            if lightCurve is None:
                return None, None

            cleanedLightCurve = lightCurve.remove_nans()
            if len(cleanedLightCurve) == 0:
                return None, {
                    "normalizationApplied": False,
                    "fluxMedian": None,
                    "fluxStd": None,
                    "fluxSnr": None,
                    "medianUnstable": True,
                    "lowSNR": True,
                    "lowQualityLightCurve": True,
                    "lowQualityReason": "No valid cadences after removing NaNs",
                    "validCadenceCount": 0,
                }

            fluxArray = np.asarray(getattr(cleanedLightCurve.flux, "value", cleanedLightCurve.flux), dtype=float)
            validMask = np.isfinite(fluxArray)

            fluxMask = getattr(cleanedLightCurve.flux, "mask", None)
            if fluxMask is not None:
                fluxMaskArray = np.asarray(fluxMask, dtype=bool)
                if fluxMaskArray.shape == ():
                    validMask &= (not bool(fluxMaskArray))
                else:
                    validMask &= np.logical_not(fluxMaskArray)

            if hasattr(cleanedLightCurve.time, "value"):
                timeValues = np.asarray(cleanedLightCurve.time.value, dtype=float)
                validMask &= np.isfinite(timeValues)

            validFlux = fluxArray[validMask]
            if validFlux.size == 0:
                return None, {
                    "normalizationApplied": False,
                    "fluxMedian": None,
                    "fluxStd": None,
                    "fluxSnr": None,
                    "medianUnstable": True,
                    "lowSNR": True,
                    "lowQualityLightCurve": True,
                    "lowQualityReason": "No valid cadences after masking invalid flux values",
                    "validCadenceCount": 0,
                }

            medianFlux = float(np.nanmedian(validFlux))
            stdFlux = float(np.nanstd(validFlux))
            medianUnstable = (not np.isfinite(medianFlux)) or abs(medianFlux) < self.normalizationStdFloor
            lowStd = (not np.isfinite(stdFlux)) or stdFlux < self.normalizationStdFloor
            fluxSnr = None if lowStd else float(abs(medianFlux) / stdFlux)
            lowSNR = fluxSnr is None or fluxSnr < self.lowSnrThreshold

            normalizationMetadata = {
                "normalizationApplied": False,
                "fluxMedian": medianFlux,
                "fluxStd": stdFlux,
                "fluxSnr": fluxSnr,
                "medianUnstable": bool(medianUnstable),
                "lowSNR": bool(lowSNR),
                "lowQualityLightCurve": bool(medianUnstable or lowSNR or lowStd),
                "lowQualityReason": None,
                "validCadenceCount": int(validFlux.size),
            }

            if lowStd:
                normalizationMetadata["lowQualityReason"] = "Flux standard deviation is too small for stable normalization"
                return cleanedLightCurve, normalizationMetadata

            standardizedFlux = np.full_like(fluxArray, np.nan, dtype=float)
            standardizedFlux[validMask] = (fluxArray[validMask] - medianFlux) / stdFlux

            standardizedLightCurve = cleanedLightCurve.copy()
            standardizedLightCurve.flux = standardizedFlux * u.dimensionless_unscaled

            if hasattr(standardizedLightCurve, "flux_err") and standardizedLightCurve.flux_err is not None:
                fluxErrArray = np.asarray(
                    getattr(standardizedLightCurve.flux_err, "value", standardizedLightCurve.flux_err),
                    dtype=float,
                )
                standardizedFluxErr = np.full_like(fluxErrArray, np.nan, dtype=float)
                finiteFluxErr = np.isfinite(fluxErrArray)
                standardizedFluxErr[finiteFluxErr] = fluxErrArray[finiteFluxErr] / stdFlux
                standardizedLightCurve.flux_err = standardizedFluxErr * u.dimensionless_unscaled

            normalizationMetadata["normalizationApplied"] = True
            return standardizedLightCurve.remove_nans(), normalizationMetadata

    def _storeLightCurve(self, lightCurve, outputFile, ticId, authorLabel):
            try:
                lightCurve.to_fits(path=outputFile, overwrite=True)
            except (AttributeError, ValueError) as exc:
                self.logger.warning(
                    "to_fits failed for TIC %s author %s, using fallback: %s",
                    ticId,
                    authorLabel,
                    exc,
                )
                try:
                    timeCol = fits.Column(name="TIME", format="D", array=lightCurve.time.jd)
                    fluxCol = fits.Column(name="FLUX", format="E", array=np.asarray(lightCurve.flux.value, dtype=float))
                    cols = fits.ColDefs([timeCol, fluxCol])
                    hdu = fits.BinTableHDU.from_columns(cols)
                    hdu.writeto(outputFile, overwrite=True)
                except Exception as fallbackExc:
                    self.logger.error(
                        "Fallback FITS write also failed for TIC %s author %s: %s",
                        ticId,
                        authorLabel,
                        fallbackExc,
                    )
                    return False

            return True

standardizer = MatchedLightCurveStandardizer()

def standardize_to_file(raw_path, output_path):
    """Use the original pipeline's reader, standardizer, and writer."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    curve = standardizer._readStoredRawLightCurve(str(raw_path), 'TESSCut')
    normalized_curve, normalization = standardizer._standardizeLightCurve(curve)
    if normalized_curve is None or normalization is None or not normalization['normalizationApplied']:
        raise RuntimeError(f'Standardization failed for {raw_path}: {normalization}')
    if not standardizer._storeLightCurve(normalized_curve, str(output_path), 'NA', 'TESSCut_standardized'):
        raise RuntimeError(f'Could not write standardized FITS: {output_path}')
    return normalization

def verify_features_against_baseline(name, extracted, saved, tag):
    extracted['VSXId'] = extracted['VSXId'].astype(str)
    extracted = extracted.set_index('VSXId', drop=False).loc[saved.index]
    extracted.to_parquet(OUTPUT_DIR / f'{name}_{tag}_features.parquet', index=False)
    if not extracted['FeatureStatus'].eq('ok').all():
        raise RuntimeError(f'{name}: {tag} extraction failed; inspect saved features')
    errors = []
    for feature in FEATURES:
        left = pd.to_numeric(saved[feature], errors='raise').to_numpy(dtype=float)
        right = pd.to_numeric(extracted[feature], errors='raise').to_numpy(dtype=float)
        for idx in np.flatnonzero(~np.isclose(left, right, rtol=FEATURE_RTOL, atol=FEATURE_ATOL, equal_nan=True)):
            errors.append({'VSXId': saved.index[idx], 'Feature': feature, 'Saved': left[idx], 'Reextracted': right[idx]})
    pd.DataFrame(errors, columns=['VSXId','Feature','Saved','Reextracted']).to_csv(OUTPUT_DIR / f'{name}_{tag}_mismatches.csv', index=False)
    if errors:
        raise RuntimeError(f'{name}: {len(errors)} {tag} feature mismatches; do not bypass this guard')
    print(name, tag, '39-feature check passed')
    return extracted


SPOC matched stars: 506
QLP matched stars: 1355


## Baseline and no-detrending validation

Each original standardized light curve is re-extracted and compared with its saved baseline features. An independent raw-FITS → standardization → extraction path, without detrending, provides a second validation of the 39 predictors. Both validation paths reproduce the saved baseline features for the SPOC- and QLP-matched cohorts, establishing that subsequent differences arise after the controlled detrending intervention rather than from the raw-data loading, standardization, FITS serialization, or feature-extraction path.


In [14]:
for name, data in matched.items():
    meta, old = data['metadata'], data['original']
    baseline = ExtractFeatures(meta.reset_index(drop=True), CASES[name]['metadata'], WorkerCount=WorkerCount)
    verify_features_against_baseline(name, baseline, old, 'original_path')
    paths = {vsx: resolve_source(row.get(rawPathColumn), CASES[name]['metadata'].parent) for vsx, row in meta.iterrows()}
    if pd.Series([build_detrended_filename(path) for path in paths.values()]).duplicated().any():
        raise RuntimeError(f'{name}: duplicate output basenames; resolve before copied writer runs')
    data['raw_paths'] = paths
    noop_dir = OUTPUT_DIR / name / 'no_detrending_standardized_fits'
    noop_meta = meta.copy()
    normalization_rows = []
    for vsx, row in meta.iterrows():
        out = noop_dir / (hashlib.sha256(vsx.encode()).hexdigest()[:20] + '_standardized.fits')
        normalization = standardize_to_file(paths[vsx], out)
        noop_meta.at[vsx, 'lightCurvePath'] = str(out)
        normalization_rows.append({'VSXId': vsx, **normalization})
    noop_meta.to_parquet(OUTPUT_DIR / f'{name}_no_detrending_metadata.parquet', index=False)
    pd.DataFrame(normalization_rows).to_csv(OUTPUT_DIR / f'{name}_no_detrending_normalization.csv', index=False)
    noop_features = ExtractFeatures(noop_meta.reset_index(drop=True), CASES[name]['metadata'], WorkerCount=WorkerCount)
    verify_features_against_baseline(name, noop_features, old, 'raw_standardized_no_detrending')


INFO:ControlledConditionalDetrending:Starting parallel feature extraction with WorkerCount=16
0% (0/9360) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/9360) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/21346) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/21346) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/6657) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/6657) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/6433) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/6433) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/24602) of the cadences will be ignored due to the quality mask (qu

SPOC original_path 39-feature check passed


Value for TARGETID is None.
0% (0/6597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/6597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/6657) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/6657) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/27654) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/27654) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/4765) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/4765) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/25241) of the cadenc

SPOC raw_standardized_no_detrending 39-feature check passed


0% (0/31244) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/31244) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/28868) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/28868) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/28826) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/33018) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/33175) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/28826) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/28434) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/42627) of the cadences will be ignored due to the quality mask (quality_bitmask=17

QLP original_path 39-feature check passed


Value for TARGETID is None.
0% (0/28868) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/28868) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/42627) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/42627) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/48327) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/48327) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/31244) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/31244) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/28434) of the 

QLP raw_standardized_no_detrending 39-feature check passed


## Conditional detrending with identical standardization

The pipeline's conditional detrending procedure and original thresholds are applied to the matched TESSCut light curves. Every successfully corrected raw FITS light curve is passed through the same pipeline standardizer and writer before feature extraction; uncorrected stars retain their original standardized paths. Original observational metadata, including original flux statistics and `QualityScore` inputs, remain fixed, while post-correction normalization statistics are retained separately for audit.

The procedure detected and detrended 228 of 506 SPOC-matched TESSCut light curves and 681 of 1,355 QLP-matched TESSCut light curves. No raw FITS files were missing and no detrending operations failed in either cohort.


In [15]:
for name, data in matched.items():
    case_dir = OUTPUT_DIR / name
    case_dir.mkdir(parents=True, exist_ok=True)
    working = data['metadata'].reset_index(drop=True).copy()
    # Resolve paths only; photometry and metadata values are otherwise unchanged.
    working[rawPathColumn] = working['VSXId'].map(lambda vsx: str(data['raw_paths'][str(vsx)]))
    input_path = case_dir / 'matched_TESSCut_QC_metadata.parquet'
    output_path = case_dir / 'matched_TESSCut_detrend_metadata.parquet'
    working.to_parquet(input_path, index=False)
    processed, processing_summary = conditional_detrend_comparison_tesscut(
        input_path, output_path, case_dir / 'detrended_fits'
    )
    bad = processed['detrendStatus'].isin(['failed', 'missing_raw_fits', 'not_eligible_qc']) | processed['trendDetectionStatus'].eq('failed')
    if bad.any():
        raise RuntimeError(f'{name}: {int(bad.sum())} processing failures; inspect {output_path}')
    use_detrended_mask = (
        processed['detrended'].fillna(False).astype(bool)
        & processed['detrendLightCurvePath'].notna()
        & processed['detrendLightCurvePath'].astype(str).str.strip().ne('')
    )
    processed['detrendRawOutputPath'] = processed['detrendLightCurvePath']
    processed['detrendStandardizedLightCurvePath'] = None
    normalization_rows = []
    for idx in processed.index[use_detrended_mask]:
        vsx = str(processed.at[idx, 'VSXId'])
        corrected_raw = Path(processed.at[idx, 'detrendLightCurvePath'])
        standardized_path = case_dir / 'detrended_standardized_fits' / (hashlib.sha256(vsx.encode()).hexdigest()[:20] + '_standardized.fits')
        normalization = standardize_to_file(corrected_raw, standardized_path)
        processed.at[idx, 'detrendStandardizedLightCurvePath'] = str(standardized_path)
        processed.at[idx, 'lightCurvePath'] = str(standardized_path)
        normalization_rows.append({'VSXId': vsx, **normalization})
    pd.DataFrame(normalization_rows, columns=['VSXId','normalizationApplied','fluxMedian','fluxStd','fluxSnr','medianUnstable','lowSNR','lowQualityLightCurve','lowQualityReason','validCadenceCount']).to_csv(case_dir / 'post_detrending_normalization.csv', index=False)
    processed.to_parquet(output_path, index=False)
    new = ExtractFeatures(processed, output_path, WorkerCount=WorkerCount)
    new['VSXId'] = new['VSXId'].astype(str)
    new = new.set_index('VSXId', drop=False).loc[data['original'].index]
    new.to_parquet(case_dir / 'TESSCut_conditional_detrended_features.parquet', index=False)
    if not new['FeatureStatus'].eq('ok').all():
        raise RuntimeError(f'{name}: conditional extraction failed; see saved FeatureStatus/FeatureError')
    if not new[TARGET].astype(str).equals(data['original'][TARGET].astype(str)):
        raise RuntimeError('Labels changed during extraction')
    unchanged = processed.set_index('VSXId')['detrended'].fillna(False).eq(False)
    ids = new.index[unchanged.reindex(new.index).to_numpy()]
    for feature in FEATURES:
        if not np.allclose(new.loc[ids,feature], data['original'].loc[ids,feature], rtol=FEATURE_RTOL, atol=FEATURE_ATOL, equal_nan=True):
            raise RuntimeError(f'{name}: uncorrected stars changed for {feature}')
    data['detrended'] = new
    data['diagnostics'] = processed
    audit = processed[['VSXId','rawLightCurvePath','lightCurvePath','detrendRawOutputPath','detrendStandardizedLightCurvePath','detrended','detrendStatus','trendDetectionStatus']]
    audit.to_csv(case_dir / 'selected_path_audit.csv', index=False)
    print(name, 'detrended stars:', int(use_detrended_mask.sum()))
    display(processed.groupby(['trendDetectionStatus','detrended'], dropna=False).size().rename('Stars'))


0% (0/21980) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/21980) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/6597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/6597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/6657) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/6657) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/27654) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/27654) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/27657) of the cadences will be ignored due to 

{'input': '/data/projects/TESS-research/summary/controlled_conditional_detrending_standardized_20261004_v2/SPOC/matched_TESSCut_QC_metadata.parquet', 'output': '/data/projects/TESS-research/summary/controlled_conditional_detrending_standardized_20261004_v2/SPOC/matched_TESSCut_detrend_metadata.parquet', 'rows': 506, 'processed': 506, 'trendDetected': 228, 'detrended': 228, 'missingRaw': 0, 'failed': 0}


Value for TARGETID is None.
0% (0/14931) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/14931) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/2424) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/2424) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/13392) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/13392) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/11033) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/11033) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/24353) of the ca

SPOC detrended stars: 228


trendDetectionStatus  detrended
drift                 True         228
no_drift              False        278
Name: Stars, dtype: int64

...

0% (0/28868) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/28868) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/28868) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/28868) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/19194) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/19194) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/30606) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/30606) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/3508) of the cadences will be ignored due 

{'input': '/data/projects/TESS-research/summary/controlled_conditional_detrending_standardized_20261004_v2/QLP/matched_TESSCut_QC_metadata.parquet', 'output': '/data/projects/TESS-research/summary/controlled_conditional_detrending_standardized_20261004_v2/QLP/matched_TESSCut_detrend_metadata.parquet', 'rows': 1355, 'processed': 1355, 'trendDetected': 681, 'detrended': 681, 'missingRaw': 0, 'failed': 0}


0% (0/17499) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/17499) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/14725) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/14725) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/9144) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/9144) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/9165) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (0/9165) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
Value for TARGETID is None.
0% (0/11608) of the cadences will be ignored due to 

QLP detrended stars: 681


trendDetectionStatus  detrended
drift                 True         681
no_drift              False        674
Name: Stars, dtype: int64

## Paired Random Forest evaluation

Official-product, original-TESSCut, and conditionally detrended-TESSCut models are evaluated on identical training and test object IDs for each split. Median imputation is fitted independently using training data only. The analysis uses seeds 42–141, preserving the canonical seed-42 split while measuring performance across 100 paired partitions. Accuracy, balanced accuracy, and macro-F1 are recorded together with the corresponding official-product performance gaps.


In [16]:
def fit_predict(table, train_ids, test_ids, seed):
    model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', RandomForestClassifier(n_estimators=500, max_depth=None, min_samples_split=2, min_samples_leaf=1, class_weight='balanced_subsample', n_jobs=-1, random_state=seed))])
    model.fit(table.loc[train_ids, FEATURES], table.loc[train_ids, TARGET].astype(str))
    return model.predict(table.loc[test_ids, FEATURES])

def metrics(y, pred):
    return {'Accuracy': accuracy_score(y,pred), 'BalancedAccuracy': balanced_accuracy_score(y,pred), 'MacroF1': f1_score(y,pred,average='macro',zero_division=0)}

performance, differences, predictions, assignments = [], [], [], []
for name, data in matched.items():
    ids = data['official'].index.to_numpy()
    for seed in SPLIT_SEEDS:
        train, test = train_test_split(ids, test_size=TEST_SIZE, stratify=data['official'].loc[ids,TARGET].astype(str), random_state=seed)
        train, test = sorted(train), sorted(test)
        y = data['official'].loc[test,TARGET].astype(str).to_numpy()
        scores, preds = {}, {}
        for variant, table in [('Official',data['official']), ('OriginalTESSCut',data['original']), ('DetrendedTESSCut',data['detrended'])]:
            preds[variant] = fit_predict(table, train, test, seed)
            scores[variant] = metrics(y,preds[variant])
            performance.append({'Comparison':name, 'SplitSeed':seed, 'Variant':variant, 'TrainStars':len(train), 'TestStars':len(test), **scores[variant]})
        if seed == 42:
            observed = [scores['Official']['Accuracy'],scores['Official']['BalancedAccuracy'],scores['OriginalTESSCut']['Accuracy'],scores['OriginalTESSCut']['BalancedAccuracy']]
            if not np.allclose(observed,CANONICAL_BASELINE[name],rtol=0,atol=5e-6): raise RuntimeError(f'{name}: canonical baseline mismatch: {observed}')
        for metric in scores['Official']:
            before = scores['Official'][metric] - scores['OriginalTESSCut'][metric]
            after = scores['Official'][metric] - scores['DetrendedTESSCut'][metric]
            differences.append({'Comparison':name,'SplitSeed':seed,'Metric':metric,'GapBefore':before,'GapAfter':after,'TESSCutImprovement':scores['DetrendedTESSCut'][metric]-scores['OriginalTESSCut'][metric],'GapReduction':before-after})
        for idx,vsx in enumerate(test): predictions.append({'Comparison':name,'SplitSeed':seed,'VSXId':vsx,'TrueFamily':y[idx],**{key: value[idx] for key,value in preds.items()}})
        assignments.extend({'Comparison':name,'SplitSeed':seed,'VSXId':vsx,'Subset':subset} for subset,items in [('train',train),('test',test)] for vsx in items)
        print(name, 'seed',seed,'complete')
        pd.DataFrame(performance).to_csv(OUTPUT_DIR/'performance.csv',index=False)
        pd.DataFrame(differences).to_csv(OUTPUT_DIR/'paired_differences.csv',index=False)
        pd.DataFrame(predictions).to_parquet(OUTPUT_DIR/'paired_predictions.parquet',index=False)
        pd.DataFrame(assignments).to_parquet(OUTPUT_DIR/'split_assignments.parquet',index=False)
performance_df, difference_df = pd.DataFrame(performance), pd.DataFrame(differences)
display(performance_df)
display(difference_df)
summary = difference_df.groupby(['Comparison','Metric']).agg(Splits=('SplitSeed','size'),MeanGapBefore=('GapBefore','mean'),MeanGapAfter=('GapAfter','mean'),MeanImprovement=('TESSCutImprovement','mean'),ImprovedSplits=('TESSCutImprovement',lambda x:int((x>0).sum())),ImprovementP025=('TESSCutImprovement',lambda x:x.quantile(.025)),ImprovementP975=('TESSCutImprovement',lambda x:x.quantile(.975))).reset_index()
summary.to_csv(OUTPUT_DIR/'summary.csv',index=False)
display(summary)
manifest = {'split_seeds':SPLIT_SEEDS,'test_size':TEST_SIZE,'features':FEATURES,'detrending_method':detrendMethod,'gap_days':gapThresholdDays,'minimum_finite_points':minFinitePoints,'minimum_segment_points':minSegmentPoints,'minimum_segment_days':minSegmentDurationDays,'drift_threshold':segmentDriftStrengthThreshold,'fraction_trigger':fractionSegmentsWithDriftThreshold,'inputs':{name:{key:str(value) for key,value in case.items()} for name,case in CASES.items()}}
(OUTPUT_DIR/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
source_notebook_hashes = {'pipeline': '445d54e6ec73aa562a45615e800658d17aad17586dd5db6b9a1840090dd53c63', 'extractor': 'b8f28dfa60a87a19f88ec6c0da6f34b1cb26a98dc01f6ddf2d079463fb62b02f'}
manifest["source_notebook_sha256"] = source_notebook_hashes
(OUTPUT_DIR/"run_manifest.json").write_text(json.dumps(manifest, indent=2))

manifest['post_detrending_standardization'] = 'Exact pipeline _standardizeLightCurve and _storeLightCurve'
manifest['no_detrending_raw_standardization_check'] = 'required for all matched TESSCut stars'
manifest['original_observational_metadata'] = 'held fixed'
(OUTPUT_DIR/'run_manifest.json').write_text(json.dumps(manifest, indent=2))


SPOC seed 42 complete
SPOC seed 43 complete
SPOC seed 44 complete
SPOC seed 45 complete
SPOC seed 46 complete
SPOC seed 47 complete
SPOC seed 48 complete
SPOC seed 49 complete
SPOC seed 50 complete
SPOC seed 51 complete
SPOC seed 52 complete
SPOC seed 53 complete
SPOC seed 54 complete
SPOC seed 55 complete
SPOC seed 56 complete
SPOC seed 57 complete
SPOC seed 58 complete
SPOC seed 59 complete
SPOC seed 60 complete
SPOC seed 61 complete
SPOC seed 62 complete
SPOC seed 63 complete
SPOC seed 64 complete
SPOC seed 65 complete
SPOC seed 66 complete
SPOC seed 67 complete
SPOC seed 68 complete
SPOC seed 69 complete
SPOC seed 70 complete
SPOC seed 71 complete
SPOC seed 72 complete
SPOC seed 73 complete
SPOC seed 74 complete
SPOC seed 75 complete
SPOC seed 76 complete
SPOC seed 77 complete
SPOC seed 78 complete
SPOC seed 79 complete
SPOC seed 80 complete
SPOC seed 81 complete
SPOC seed 82 complete
SPOC seed 83 complete
SPOC seed 84 complete
SPOC seed 85 complete
SPOC seed 86 complete
SPOC seed 

,Comparison,SplitSeed,Variant,TrainStars,TestStars,Accuracy,BalancedAccuracy,MacroF1
0,SPOC,42,Official,354,152,0.763158,0.570680,0.572913
1,SPOC,42,OriginalTESSCut,354,152,0.631579,0.386845,0.367011
2,SPOC,42,DetrendedTESSCut,354,152,0.651316,0.422229,0.416909
3,SPOC,43,Official,354,152,0.730263,0.605143,0.621108
4,SPOC,43,OriginalTESSCut,354,152,0.677632,0.415907,0.396378
...,...,...,...,...,...,...,...,...
595,QLP,140,OriginalTESSCut,948,407,0.525799,0.278826,0.275970
596,QLP,140,DetrendedTESSCut,948,407,0.481572,0.239255,0.237886
597,QLP,141,Official,948,407,0.673219,0.452607,0.474467
598,QLP,141,OriginalTESSCut,948,407,0.557740,0.299686,0.308505


,Comparison,SplitSeed,Metric,GapBefore,GapAfter,TESSCutImprovement,GapReduction
0,SPOC,42,Accuracy,0.131579,0.111842,0.019737,0.019737
1,SPOC,42,BalancedAccuracy,0.183835,0.148451,0.035384,0.035384
2,SPOC,42,MacroF1,0.205903,0.156005,0.049898,0.049898
3,SPOC,43,Accuracy,0.052632,0.138158,-0.085526,-0.085526
4,SPOC,43,BalancedAccuracy,0.189236,0.231341,-0.042105,-0.042105
...,...,...,...,...,...,...,...
595,QLP,140,BalancedAccuracy,0.154661,0.194232,-0.039571,-0.039571
596,QLP,140,MacroF1,0.181008,0.219092,-0.038084,-0.038084
597,QLP,141,Accuracy,0.115479,0.164619,-0.049140,-0.049140
598,QLP,141,BalancedAccuracy,0.152921,0.192541,-0.039621,-0.039621


,Comparison,Metric,Splits,MeanGapBefore,MeanGapAfter,MeanImprovement,ImprovedSplits,ImprovementP025,ImprovementP975
0,QLP,Accuracy,100,0.150737,0.197248,-0.046511,1,-0.082371,-0.015909
1,QLP,BalancedAccuracy,100,0.162523,0.206436,-0.043913,0,-0.070495,-0.019926
2,QLP,MacroF1,100,0.185618,0.226380,-0.040762,0,-0.072288,-0.015683
3,SPOC,Accuracy,100,0.136579,0.142697,-0.006118,36,-0.059539,0.049507
4,SPOC,BalancedAccuracy,100,0.213494,0.216012,-0.002517,42,-0.045834,0.054515
5,SPOC,MacroF1,100,0.238891,0.240582,-0.001691,42,-0.057390,0.069400


3007

## Results and interpretation

Conditional detrending with consistent pipeline standardization did not reduce the official-product classification gap on average across the 100 paired splits.

For the SPOC-matched cohort, the mean TESSCut change after detrending was −0.0061 in accuracy, −0.0025 in balanced accuracy, and −0.0017 in macro-F1. Accuracy improved in 36 of 100 splits, while balanced accuracy and macro-F1 each improved in 42 of 100 splits. The mean official–TESSCut accuracy gap changed from 0.1366 before detrending to 0.1427 after detrending.

For the QLP-matched cohort, detrending reduced TESSCut performance consistently: mean changes were −0.0465 in accuracy, −0.0439 in balanced accuracy, and −0.0408 in macro-F1. Accuracy improved in 1 of 100 splits, while balanced accuracy and macro-F1 improved in none. The mean official–TESSCut accuracy gap increased from 0.1507 to 0.1972.

These results show that this implemented conditional detrending procedure, when followed by the same standardization used by the pipeline, does not account for the observed SPOC/QLP–TESSCut performance differences and does not narrow those gaps. The experiment isolates this specific detrending intervention; it does not establish the physical origin of the provenance effect or evaluate every possible detrending method.

Standardization parameters are recalculated for each corrected curve using the original formula, while original observational metadata remain fixed. The 100 paired splits quantify variation across partitions of the matched cohorts.
